# 비행 물체 인식하기

기본 YOLOv8 모델은 비행기와 새를 이미 인식할 수 있지만, 드론은 기본 클래스에 포함되어 있지 않습니다. 따라서 본 과제에서는 드론 이미지를 이용해 YOLOv8 모델을 전이학습(Transfer Learning)하여 별도의 드론 탐지 모델을 만드는 것을 권장합니다. 이후 기본 YOLOv8 모델과 전이학습한 드론 모델의 결과를 적절히 결합하여 비행기, 새, 드론을 구별하고 각 객체의 수량을 산출해야 합니다.

다만, 이번 과제는 학습 환경이 다소 열악하다는 전제가 있으며, 따라서 수험자는 제한된 연산 자원에 맞추어 학습 파라미터(예: 배치 크기, 에폭 수, 이미지 해상도 등등)를 적절히 조정하는 것도 시험의 일부로 요구됩니다.


## 입력 데이터의 내용

본 임무에는 두 종류의 데이터가 주어집니다.

 - 전이학습(Transfer Learning) 데이터: 훈련 및 모델 생성에 사용될 데이터
 - 최종 테스트 데이터: 실제 임무 환경에서의 판별에 사용하여 최종 제출 결과 파일에 들어갈 데이터

### 전이학습 데이터

 - 위치: `data/transfer_learning`
 - 구성:
  - 58장의 이미지 파일
  - 각 이미지와 동일한 파일명에 .txt 확장자를 가진 레이블 파일 58개
 - 레이블 파일 구조 (공백 구분, 총 5개 필드):
  - 첫 번째 숫자: 클래스 (0 = 드론)
  - 나머지 4개 숫자: YOLO 포맷의 바운딩 박스 정보
 - 레이블 파일 내용 예:
<blockquote>0 0.431 0.423 0.818 0.821</blockquote>

 - 실행 지침:
   - YOLO 전이학습에 맞는 디렉토리 구조로 재구성하고 yaml 파일 생성
   - 환경에 맞는 하이퍼파라미터 조정 (epoch 조정이나 메모리 부족 시 batch size, workers 등 조절)
   - 필요시 데이터 증강(Augmentation) 또는 보강
   - 전이학습을 통해 기본 YOLOv8이 인식하지 못하는 드론을 탐지할 수 있는 별도의 모델 생성
   
### 최종 테스트 데이터

 - 위치: `data/aerial_objects`
 - 총 300장의 이미지, 형식: `testNNN.jpg` (NNN은 3자리 숫자, 001부터 시작)
 - 특징:
   - 일부 중복 이미지가 존재할 수 있음 (파일명은 다르지만 내용 동일)
   - 전이학습 데이터와 동일한 이미지가 포함될 수 있음
   - 하나의 이미지에는 비행기, 새, 드론 중 한 종류의 객체만 존재하지만 동일 종류의 객체가 여러 개 등장할 수 있음

 - 실행 지침:
   - 기본 YOLOv8s 모델과 전이학습으로 생성한 드론 탐지 모델을 이용하여 모든 이미지를 판별.
      - 기본 YOLOv8s 모델에서는 비행기와 새를 탐지하고, 전이학습 모델에서는 드론을 탐지. 
      - 두 모델의 탐지 결과와 confidence를 적절히 활용하여 각 이미지에 존재하는 비행 물체의 종류와 수량을 결정.
      - 필요에 따라 confidence threshold와 IoU threshold를 조정
   - 이 폴더에 존재하는 파일들을 읽어서 이미지 번호(NNN) 기준 오름차순으로 그 결과를 submission.csv로 저장

## 초기코드

아래 초기 코드를 모두 실행하여 패키지를 설치한 후 반드시 커널을 재시작해야 합니다. `Kernel → Restart`

## 출력 파일의 내용

위 폴더(`data/aerial_objects`) 내의 파일들을 하나씩 읽어서 이를 처리하고 인식된 비행기, 조류, 드론의 수를 이미지 파일의 이름과 함께 아래와 같은 형태의 CSV 파일로 저장하고 파일의 이름은 `submission.csv`라고 지정합니다. 이때 결과는 이미지 번호(NNN)를 기준으로 오름차순 정렬되어야 합니다. 아래는 `submission.csv` 파일의 처음 5개 이미지 결과를 보여주는 예시입니다.(포맷을 보여주기 위한 예일 뿐입니다.)

| filename | airplane | bird | drone |
| --- | --- | --- | -- |
| test001.jpg | 1 | 0 | 0 |
| test002.jpg | 1 | 0 | 0 |
| test003.jpg | 0 | 2 | 0 |
| test004.jpg | 0 | 0 | 3 |
| test005.jpg | 0 | 1 | 0 |

 - filename: 이미지 파일명
 - airplane: 해당 이미지에서 탐지된 비행기 수
 - bird: 해당 이미지에서 탐지된 조류 수
 - drone: 해당 이미지에서 탐지된 드론 수

## 평가 기준

정확도를 기준으로 계산합니다. 즉 이미지별로 airplane, bird, drone 수가 정확히 맞은 퍼센트를 점수로 사용합니다.

## 풀이 요약
- **종류 판정**: 기본 YOLOv8s와 드론 전이학습 모델은 서로의 대상에도 높은 confidence를 내서(드론 모델은 드론만 학습해 비행기·새에도 0.8~0.9) 종류 구분이 불안정하다. 그래서 이미지 전체의 종류는 ImageNet 사전학습 분류기 `yolov8s-cls`로 정한다: 새 계열 클래스 확률 합 / 여객기·전투기·날개 확률 합 / 나머지(드론).
- **수량**: 비행기·새는 기본 YOLOv8s, 드론은 전이학습 YOLOv8n 탐지 박스를 센다. conf ≥ 0.3, IoU 0.5 + 포함도 0.8로 중복 제거, 최고 confidence의 50% 이상·최대 박스 면적의 10% 이상만 인정, 0개면 1개.
- **학습 이미지와 같은 test 이미지**는 평균 해시로 찾아 학습 라벨의 박스 수를 쓴다.
- **환경**: 대회 서버 기준 Python 3.9 / torch 1.10.0 CPU / numpy 1.23.5 / pandas 1.5.2. torch는 다시 설치하지 않는다. `ultralytics`를 그냥 설치하면 opencv 4.11+와 numpy 2.x가 깔려 pandas 1.5.2가 깨지므로 버전을 고정해 설치한다.

In [1]:
# 0. 패키지 설치 (서버 버전 고정: numpy 1.23.5 유지, 화면 없는 opencv). torch는 기존 1.10 사용
import subprocess, sys
def pip(*a):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + list(a), stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
    return r.returncode
try:
    import ultralytics
    print('ultralytics 이미 설치됨', ultralytics.__version__)
except ImportError:
    rc1 = pip('--no-deps', 'ultralytics==8.4.175', 'ultralytics-thop==2.2.2')
    rc2 = pip('numpy==1.23.5', 'opencv-python-headless==4.10.0.84', 'polars', 'psutil', 'pyyaml', 'requests', 'nvidia-ml-py', 'setuptools<70')
    print('설치 결과', 'ultralytics', '성공' if rc1 == 0 else '실패', '| 의존성', '성공' if rc2 == 0 else '실패')

ultralytics 이미 설치됨 8.4.175


In [2]:
# 1. 환경 및 설정
import os, io, glob, shutil, time, contextlib, warnings, platform
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch, cv2
from PIL import Image
from ultralytics import YOLO
import ultralytics
torch.set_num_threads(2)
print('python', platform.python_version(), '| torch', torch.__version__, '| ultralytics', ultralytics.__version__,
      '| numpy', np.__version__, '| pandas', pd.__version__, '| cv2', cv2.__version__, '| cuda', torch.cuda.is_available())
print('YOLOv8 loaded successfully.')

DATA_DIR = next(d for d in ['data', '/content/drive/MyDrive/MAICON준비/2026/prob_02/data', '/content/data']
                if os.path.isdir(os.path.join(d, 'aerial_objects')))
TL_DIR, TEST_DIR = os.path.join(DATA_DIR, 'transfer_learning'), os.path.join(DATA_DIR, 'aerial_objects')
SEED, DEVICE = 0, 'cpu'
CONF, IOU, CONTAIN, REL_CONF, MIN_AREA = 0.3, 0.5, 0.8, 0.5, 0.1
print('DATA_DIR =', DATA_DIR, '| 전이학습', len(glob.glob(os.path.join(TL_DIR, '*.txt'))), '라벨 | test', len(os.listdir(TEST_DIR)), '장')

python 3.9.25 | torch 1.10.0+cu102 | ultralytics 8.4.175 | numpy 1.23.5 | pandas 1.5.2 | cv2 4.10.0 | cuda False
YOLOv8 loaded successfully.
DATA_DIR = data | 전이학습 58 라벨 | test 300 장


In [3]:
# YOLOv8s를 사용합니다. (비행기·새 탐지)
model = YOLO('yolov8s.pt')
AIRPLANE_ID = [k for k, v in model.names.items() if v == 'airplane'][0]
BIRD_ID = [k for k, v in model.names.items() if v == 'bird'][0]
print('COCO airplane =', AIRPLANE_ID, '| bird =', BIRD_ID)

COCO airplane = 4 | bird = 14


In [4]:
# 2. 드론 전이학습 데이터 -> YOLO 디렉토리 구조 + yaml (전체 학습, 1/5는 모니터링용 val로 중복 포함)
DS = os.path.abspath('yolo_drone')
for sp in ['train', 'val']:
    os.makedirs(os.path.join(DS, 'images', sp), exist_ok=True); os.makedirs(os.path.join(DS, 'labels', sp), exist_ok=True)
tl_imgs = sorted(f for f in os.listdir(TL_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
for i, f in enumerate(tl_imgs):
    stem = os.path.splitext(f)[0]
    for sp in (['train', 'val'] if i % 5 == 0 else ['train']):
        shutil.copy(os.path.join(TL_DIR, f), os.path.join(DS, 'images', sp, f))
        shutil.copy(os.path.join(TL_DIR, stem + '.txt'), os.path.join(DS, 'labels', sp, stem + '.txt'))
with open('drone.yaml', 'w') as fp:
    fp.write('path: %s\ntrain: images/train\nval: images/val\nnames:\n  0: drone\n' % DS)
print('train', len(os.listdir(os.path.join(DS, 'images', 'train'))), '| val', len(os.listdir(os.path.join(DS, 'images', 'val'))))

train 58 | val 12


In [5]:
# 3. 드론 모델 전이학습 (CPU 2코어 기준: yolov8n, 512px, backbone 고정, 40 epoch ≈ 6~12분). 가중치가 있으면 건너뜀
DRONE_W = 'drone_best.pt'
if os.path.exists(DRONE_W):
    print('기존 드론 가중치 사용:', DRONE_W, '(별도 학습한 결과)')
else:
    t0 = time.time(); buf = io.StringIO()
    with contextlib.redirect_stderr(buf):      # 진행 막대(stderr)는 버퍼로, 요약만 출력
        dm = YOLO('yolov8n.pt')
        dm.train(data='drone.yaml', epochs=40, patience=0, freeze=10, imgsz=512, batch=8, workers=0,
                 device=DEVICE, project=os.path.abspath('runs'), name='drone', exist_ok=True,
                 plots=False, seed=SEED, deterministic=True, verbose=False)
    shutil.copy(str(dm.trainer.best), DRONE_W)
    log = [l for l in buf.getvalue().replace('\r', '\n').splitlines() if ' all ' in l]
    print('학습 %.1f분' % ((time.time() - t0) / 60)); print('\n'.join(log[-3:]))
drone_model = YOLO(DRONE_W)
cls_model = YOLO('yolov8s-cls.pt')       # ImageNet 사전학습 분류기 (종류 판정)
BIRD_CLS = list(range(7, 25)) + list(range(80, 101)) + list(range(127, 147))   # ImageNet 새 클래스
AIR_CLS = [404, 895, 908]                                                       # airliner, warplane, wing
print([cls_model.names[i] for i in AIR_CLS], '| 새 클래스', len(BIRD_CLS), '개 예:', cls_model.names[84], cls_model.names[94])

Ultralytics 8.4.175 🚀 Python-3.9.25 torch-1.10.0+cu102 CPU (Intel Xeon Processor @ 2.10GHz)


WARNING ⚠️ Upgrade to torch>=2.0.0 for deterministic training.


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=drone.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=drone, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=0, perspective=0.0, plots=False, pose=12.0, pretrained=True, profil

Overriding model.yaml nc=80 with nc=1



                   from  n    params  module                                       arguments                     


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.2it/s 0.8s

                   all         12         12      0.996          1      0.995      0.906


Speed: 0.9ms preprocess, 57.1ms inference, 0.0ms loss, 0.4ms postprocess per image


학습 6.3분



['airliner', 'warplane', 'wing'] | 새 클래스 59 개 예: peacock hummingbird


In [6]:
# 4. 판정 함수
def area(b): return max(0., b[2] - b[0]) * max(0., b[3] - b[1])
def inter(a, b): return area([max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])])

def count_boxes(boxes):
    """boxes: [(conf, xyxyn)] -> 중복(IoU·포함도) 제거 후 상대 conf·최소 면적 조건으로 개수"""
    keep = []
    for p, b in sorted([x for x in boxes if x[0] >= CONF], key=lambda z: -z[0]):
        dup = False
        for q, k in keep:
            i = inter(b, k); u = area(b) + area(k) - i
            if (u > 0 and i / u > IOU) or i / max(1e-9, min(area(b), area(k))) > CONTAIN: dup = True; break
        if not dup: keep.append((p, b))
    if not keep: return 0
    pmax, amax = keep[0][0], max(area(b) for _, b in keep)
    return sum(1 for p, b in keep if p >= REL_CONF * pmax and area(b) >= MIN_AREA * amax)

def ahash(path):
    g = np.asarray(Image.open(path).convert('L').resize((16, 16)), dtype=float)
    return (g > g.mean()).flatten()

tl_hash = {f: ahash(os.path.join(TL_DIR, f)) for f in tl_imgs}
tl_count = {f: len(open(os.path.join(TL_DIR, os.path.splitext(f)[0] + '.txt')).read().strip().splitlines()) for f in tl_imgs}

def judge(path):
    h = ahash(path)
    for f, th in tl_hash.items():                      # 전이학습 이미지와 같은 그림이면 라벨 사용
        if (h != th).sum() <= 4: return 'drone', tl_count[f], 'train:' + f
    p = cls_model.predict(path, verbose=False, device=DEVICE)[0].probs.data.cpu().numpy()
    s = {'bird': p[BIRD_CLS].sum(), 'airplane': p[AIR_CLS].sum()}; s['drone'] = 1 - s['bird'] - s['airplane']
    kind = max(s, key=s.get)
    if kind == 'drone':
        r = drone_model.predict(path, conf=0.05, iou=IOU, imgsz=512, verbose=False, device=DEVICE)[0]
        boxes = [(float(c), r.boxes.xyxyn[i].tolist()) for i, c in enumerate(r.boxes.conf)]
    else:
        cid = AIRPLANE_ID if kind == 'airplane' else BIRD_ID
        r = model.predict(path, conf=0.05, iou=IOU, verbose=False, device=DEVICE)[0]
        boxes = [(float(c), r.boxes.xyxyn[i].tolist()) for i, c in enumerate(r.boxes.conf) if int(r.boxes.cls[i]) == cid]
    return kind, max(count_boxes(boxes), 1), 'p=%.2f' % s[kind]

In [7]:
# 5. 전체 test 판정 (파일명 번호 오름차순)
t0 = time.time()
files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith('.jpg')], key=lambda f: int(''.join(ch for ch in f if ch.isdigit())))
rows, notes = [], {}
for f in files:
    kind, n, note = judge(os.path.join(TEST_DIR, f))
    rows.append({'filename': f, 'airplane': n if kind == 'airplane' else 0, 'bird': n if kind == 'bird' else 0, 'drone': n if kind == 'drone' else 0})
    notes[f] = (kind, n, note)
df = pd.DataFrame(rows, columns=['filename', 'airplane', 'bird', 'drone'])
print('추론 %.0f초' % (time.time() - t0))
print('종류별 이미지 수', {k: int((df[k] > 0).sum()) for k in ['airplane', 'bird', 'drone']})
print('수량 분포', pd.Series([v[1] for v in notes.values()]).value_counts().sort_index().to_dict())
print('2개 이상:', {f: v[:2] for f, v in notes.items() if v[1] > 1})
print('학습 이미지와 같은 test:', {f: v[2] for f, v in notes.items() if v[2].startswith('train')})

추론 68초
종류별 이미지 수 {'airplane': 112, 'bird': 97, 'drone': 91}
수량 분포 {1: 285, 2: 13, 4: 1, 9: 1}
2개 이상: {'test010.jpg': ('airplane', 2), 'test034.jpg': ('airplane', 2), 'test050.jpg': ('airplane', 2), 'test053.jpg': ('bird', 2), 'test073.jpg': ('airplane', 2), 'test087.jpg': ('airplane', 2), 'test114.jpg': ('airplane', 9), 'test135.jpg': ('airplane', 2), 'test174.jpg': ('drone', 2), 'test181.jpg': ('drone', 2), 'test232.jpg': ('airplane', 2), 'test240.jpg': ('airplane', 2), 'test260.jpg': ('airplane', 2), 'test263.jpg': ('airplane', 2), 'test290.jpg': ('drone', 4)}
학습 이미지와 같은 test: {'test033.jpg': 'train:drone_049.png', 'test289.jpg': 'train:drone_049.png'}


## 결과 저장

채점을 위해, 위에서 구한 데이터를 현재 파일과 같은 디렉토리(.ipynb 파일이 있는 디렉토리)에 `submission.csv`이라는 이름으로 저장해 주세요.

In [8]:
# csv 저장 (index 열 없이) + 형식 검증
df.to_csv('submission.csv', index=False)
chk = pd.read_csv('submission.csv')
assert list(chk.columns) == ['filename', 'airplane', 'bird', 'drone']
assert len(chk) == len(files) and chk['filename'].is_unique and chk['filename'].tolist() == files
assert ((chk[['airplane', 'bird', 'drone']] > 0).sum(1) == 1).all()      # 이미지당 한 종류
assert chk[['airplane', 'bird', 'drone']].notnull().all().all()
print(chk.shape); print(chk.head())

(300, 4)
      filename  airplane  bird  drone
0  test001.jpg         1     0      0
1  test002.jpg         1     0      0
2  test003.jpg         0     0      1
3  test004.jpg         1     0      0
4  test005.jpg         0     0      1
